# Ejercicio 4 — Análisis exploratorio con DuckDB

Este notebook ejecuta las consultas de `sql/02_analisis_exploratorio.sql` **por nombre**, directamente sobre los
archivos Parquet (a través de las vistas de `sql/00_vistas.sql`), y genera las gráficas usadas en
`docs/ejercicio4_analisis_exploratorio.md`. Las figuras también se guardan en `docs/img/`.

* Fuente: `data/raw/yellow/*/*.parquet`, `data/raw/green/*/*.parquet`, `data/raw/taxi_zone_lookup.csv`.
* Las consultas no dependen del año: los resultados guardados en este notebook corresponden a **2026 (enero–agosto)**,
  los únicos datos disponibles al momento del ejercicio. Si se ejecuta después del Ejercicio 5 incluirá 2024.

In [1]:
import os, sys
from pathlib import Path

# Trabajar siempre desde la raiz del proyecto (las rutas de las vistas son relativas a ella).
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "scripts")

import matplotlib.pyplot as plt
import pandas as pd
from lab_db import conectar, cargar_consultas

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

con = conectar()                       # DuckDB en memoria + vistas sobre los Parquet
Q = cargar_consultas("02_analisis_exploratorio.sql")
IMG = Path("docs/img"); IMG.mkdir(parents=True, exist_ok=True)
COLORES = {"yellow": "#E0A100", "green": "#2E8B57"}

def q(nombre):
    return con.sql(Q[nombre]).df()

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(IMG / f"{nombre}.png", dpi=110)

print(f"{len(Q)} consultas cargadas:", ", ".join(Q))
con.sql("SELECT tipo_taxi, anio, count(DISTINCT mes) AS meses, count(*) AS registros FROM viajes GROUP BY ALL ORDER BY ALL").df()

16 consultas cargadas: p1_viajes_por_mes, p2_hora_dia_semana, p3_caracteristicas_viaje, p4_velocidad_por_hora, p5a_borough_origen, p5b_top_zonas_origen, p6_aeropuertos, p7_metodo_pago, p8a_propinas_por_metodo, p8b_histograma_pct_propina, p9_composicion_total, p9b_inconsistencia_por_proveedor, p10a_distribucion_total, p10b_histograma_total, p11_impacto_limpieza, p12_nulos_passenger_count


,tipo_taxi,anio,meses,registros
0,green,2026,8,337114
1,yellow,2026,8,29703355


## Calidad de datos previa al análisis (P11, P12)

Antes de responder preguntas de comportamiento se cuantifica cuánto elimina cada regla de `viajes_limpios`
(reglas R1–R4 definidas en `sql/00_vistas.sql`, derivadas del Ejercicio 3).

In [2]:
q("p11_impacto_limpieza")

,tipo_taxi,registros,r1_fuera_de_periodo,r2_duracion_invalida,r3_distancia_invalida,r4_monto_negativo,excluidos_total,pct_excluidos
0,yellow,29703355,146,380268,953454,161970,1468439,4.94
1,green,337114,98,1453,12284,1025,14130,4.19


In [3]:
q("p12_nulos_passenger_count")

,tipo_taxi,vendor_id,payment_type,viajes,passenger_count_nulo,pct_nulo,ratecode_nulo
0,yellow,1,0,895381,895381,100.0,895381
1,yellow,1,1,4028246,0,0.0,0
2,yellow,1,2,447521,0,0.0,0
3,yellow,1,3,59767,0,0.0,0
4,yellow,1,4,36154,0,0.0,0
5,yellow,2,0,6761917,6761917,100.0,6761917
6,yellow,2,1,14592874,0,0.0,0
7,yellow,2,2,2217837,0,0.0,0
8,yellow,2,3,35455,0,0.0,0
9,yellow,2,4,201691,0,0.0,0


## P1. Volumen mensual de viajes

In [4]:
p1 = q("p1_viajes_por_mes")
p1["periodo"] = pd.to_datetime(dict(year=p1.anio, month=p1.mes, day=1))
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
for eje, (tipo, df) in zip(ejes, p1.groupby("tipo_taxi", sort=False)):
    eje.plot(df.periodo, df.viajes_por_dia, marker="o", color=COLORES[tipo])
    eje.set_title(f"{tipo}: viajes promedio por día")
    eje.set_ylim(0, df.viajes_por_dia.max() * 1.15)
    eje.tick_params(axis="x", rotation=45)
guardar(fig, "ej4_p1_viajes_por_mes")
p1

,tipo_taxi,anio,mes,viajes,viajes_por_dia,periodo
0,green,2026,1,38759,1250.0,2026-01-01
1,green,2026,2,35777,1278.0,2026-02-01
2,green,2026,3,42490,1371.0,2026-03-01
3,green,2026,4,42349,1412.0,2026-04-01
4,green,2026,5,43086,1390.0,2026-05-01
5,green,2026,6,42430,1414.0,2026-06-01
6,green,2026,7,39393,1271.0,2026-07-01
7,green,2026,8,38700,1248.0,2026-08-01
8,yellow,2026,1,3516640,113440.0,2026-01-01
9,yellow,2026,2,3210401,114657.0,2026-02-01


## P2. Demanda por hora y día de la semana

In [5]:
p2 = q("p2_hora_dia_semana")
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
fig, ejes = plt.subplots(1, 2, figsize=(13, 3.8))
for eje, tipo in zip(ejes, ["yellow", "green"]):
    m = p2[p2.tipo_taxi == tipo].pivot(index="dia_semana", columns="hora", values="pct_viajes")
    im = eje.imshow(m, aspect="auto", cmap="viridis")
    eje.set_yticks(range(7), dias); eje.set_xticks(range(0, 24, 2))
    eje.set_xlabel("hora de inicio"); eje.set_title(f"{tipo}: % de viajes")
    fig.colorbar(im, ax=eje)
guardar(fig, "ej4_p2_hora_dia")
p2.groupby(["tipo_taxi", "dia_semana"]).pct_viajes.sum().unstack(0).round(2)

tipo_taxi,green,yellow
dia_semana,,
1,14.28,11.89
2,15.13,13.63
3,15.70,14.52
4,16.54,15.79
5,14.99,14.96
6,12.00,15.95
7,11.36,13.27


## P3. Perfil del viaje típico

In [6]:
q("p3_caracteristicas_viaje")

,tipo_taxi,viajes,mediana_distancia_mi,p90_distancia_mi,mediana_duracion_min,p90_duracion_min,mediana_velocidad_mph,prom_pasajeros,pct_un_pasajero,mediana_tarifa,mediana_total
0,yellow,28234916,1.93,8.70,14.1,33.5,9.3,1.25,82.3,15.6,23.58
1,green,322984,2.14,7.51,13.3,32.6,10.1,1.30,82.8,13.5,20.52


## P4. Velocidad a lo largo del día

In [7]:
p4 = q("p4_velocidad_por_hora")
fig, eje = plt.subplots(figsize=(8, 3.5))
for tipo, df in p4.groupby("tipo_taxi"):
    eje.plot(df.hora, df.mediana_velocidad_mph, marker="o", label=tipo, color=COLORES[tipo])
eje.set_xlabel("hora de inicio"); eje.set_ylabel("mph (mediana)"); eje.set_xticks(range(0, 24, 2))
eje.set_title("Velocidad mediana por hora"); eje.legend()
guardar(fig, "ej4_p4_velocidad_hora")
p4.pivot(index="hora", columns="tipo_taxi", values=["mediana_velocidad_mph", "mediana_distancia_mi"])

mediana_velocidad_mph        mediana_distancia_mi       
tipo_taxi                 green yellow                green yellow
hora                                                              
0                         12.14  11.82                 2.29   2.60
1                         12.50  12.30                 2.33   2.56
2                         14.10  12.90                 3.32   2.62
3                         14.69  13.81                 3.28   2.85
4                         16.64  15.38                 4.39   3.50
5                         16.18  16.03                 4.60   3.71
6                         12.84  13.81                 2.10   2.93
7                         10.48  11.09                 2.08   2.29
8                          9.48   9.30                 2.09   2.00
9                          9.87   8.85                 2.19   1.84
10                        10.06   8.58                 2.30   1.78
11                         9.86   8.15                 2.31   1.72
12                         9.82   8.08                 2.30   1.70
13                         9.56   8.10                 2.20   1.74
14                         9.09   8.02                 2.17   1.78
15                         9.00   7.86                 2.05   1.73
16                         9.19   8.00                 2.05   1.70
17                         9.33   8.03                 2.00   1.65
18                         9.76   8.34                 1.95   1.63
19                        10.48   8.93                 1.93   1.72
20                        10.83   9.75                 2.03   2.01
21                        11.46  10.07                 2.17   2.13
22                        12.17  10.46                 2.30   2.29
23                        12.23  11.17                 2.36   2.49

## P5. Origen de los viajes: yellow vs green

In [8]:
p5a = q("p5a_borough_origen")
m = p5a.pivot(index="borough", columns="tipo_taxi", values="pct_viajes").fillna(0)
m = m.loc[m.max(axis=1) > 0.5].sort_values("yellow")
fig, eje = plt.subplots(figsize=(7, 3.2))
m[["yellow", "green"]].plot.barh(ax=eje, color=[COLORES["yellow"], COLORES["green"]])
eje.set_xlabel("% de viajes del tipo"); eje.set_title("Borough de origen")
guardar(fig, "ej4_p5_borough")
p5a

,tipo_taxi,borough,viajes,pct_viajes
0,yellow,Manhattan,24462277,86.64
1,yellow,Queens,2497857,8.85
2,yellow,Brooklyn,1016730,3.60
3,yellow,Bronx,218722,0.77
4,yellow,Unknown,29452,0.10
5,yellow,N/A,6294,0.02
6,yellow,Staten Island,2671,0.01
7,yellow,EWR,913,0.00
8,green,Manhattan,192082,59.47
9,green,Queens,71170,22.04


In [9]:
q("p5b_top_zonas_origen")

,tipo_taxi,borough,zona,viajes,pct_viajes
0,yellow,Manhattan,Upper East Side South,1252954,4.44
1,yellow,Manhattan,Midtown Center,1171748,4.15
2,yellow,Queens,JFK Airport,1125096,3.98
3,yellow,Manhattan,Upper East Side North,1113386,3.94
4,yellow,Manhattan,Penn Station/Madison Sq West,868701,3.08
5,yellow,Manhattan,Midtown East,864694,3.06
6,yellow,Manhattan,Times Sq/Theatre District,817549,2.90
7,yellow,Manhattan,Lincoln Square East,811841,2.88
8,yellow,Manhattan,East Village,756512,2.68
9,yellow,Manhattan,Murray Hill,737780,2.61


## P6. Viajes de aeropuerto

In [10]:
q("p6_aeropuertos")

,tipo_taxi,viajes,viajes_aeropuerto,pct_aeropuerto,total_prom_aeropuerto,total_prom_resto
0,yellow,28234916,2304726,8.16,77.94,26.00
1,green,322984,10784,3.34,48.40,24.59


## P7. Métodos de pago

In [11]:
p7 = q("p7_metodo_pago")
p7["periodo"] = p7.anio.astype(str) + "-" + p7.mes.astype(str).str.zfill(2)
fig, ejes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for eje, tipo in zip(ejes, ["yellow", "green"]):
    m = p7[p7.tipo_taxi == tipo].pivot(index="periodo", columns="metodo_pago", values="pct_viajes").fillna(0)
    m = m[m.sum().sort_values(ascending=False).index]
    m.plot.bar(stacked=True, ax=eje, width=0.85, legend=True)
    eje.set_title(f"{tipo}: mezcla de pagos (%)"); eje.set_xlabel("")
    eje.legend(fontsize=7, loc="lower right")
guardar(fig, "ej4_p7_metodo_pago")
p7.pivot_table(index=["tipo_taxi", "periodo"], columns="metodo_pago", values="pct_viajes").round(1)

metodo_pago        Desconocido  Disputa  Efectivo  Flex fare  NULL (no registrado)  Sin cargo  Tarjeta
tipo_taxi periodo                                                                                     
green     2026-01          NaN      0.2      20.6        NaN                  13.4        0.6     65.1
          2026-02          NaN      0.2      20.2        NaN                  14.4        0.6     64.5
          2026-03          NaN      0.2      19.5        NaN                  15.1        0.5     64.7
          2026-04          NaN      0.2      19.3        NaN                  14.2        0.5     65.7
          2026-05          NaN      0.2      19.8        NaN                  12.8        0.4     66.6
          2026-06          NaN      0.2      19.1        NaN                  14.6        0.4     65.6
          2026-07          NaN      0.2      18.3        NaN                  15.6        0.4     65.4
          2026-08          NaN      0.2      19.8        NaN                  15.5        0.4     64.0
yellow    2026-01          NaN      1.5       8.4       29.2                   NaN        0.4     60.4
          2026-02          NaN      1.1       8.0       30.1                   NaN        0.4     60.4
          2026-03          NaN      0.8       8.9       23.9                   NaN        0.3     66.0
          2026-04          NaN      0.6       9.4       20.9                   NaN        0.3     68.8
          2026-05          NaN      0.6       9.1       23.4                   NaN        0.3     66.7
          2026-06          0.0      0.6       9.3       26.4                   NaN        0.3     63.4
          2026-07          NaN      0.6       9.8       27.5                   NaN        0.3     61.8
          2026-08          NaN      0.7       9.9       27.6                   NaN        0.3     61.5

## P8. Propinas

In [12]:
q("p8a_propinas_por_metodo")

,tipo_taxi,metodo_pago,viajes,pct_con_propina,propina_prom,mediana_pct_propina_si_hay
0,yellow,Tarjeta,18455506,91.13,4.26,20.0
1,yellow,Efectivo,2558843,0.01,0.00,20.0
2,green,Tarjeta,211620,91.39,3.79,20.0
3,green,Efectivo,62591,0.00,0.00,NaN


In [13]:
p8b = q("p8b_histograma_pct_propina")
fig, eje = plt.subplots(figsize=(9, 3.5))
ancho = 0.4
for i, tipo in enumerate(["yellow", "green"]):
    df = p8b[p8b.tipo_taxi == tipo]
    eje.bar(df.pct_propina + (i - 0.5) * ancho, df.pct_viajes, width=ancho, label=tipo, color=COLORES[tipo])
eje.set_xlabel("propina como % de lo cobrado (pagos con tarjeta)"); eje.set_ylabel("% de viajes")
eje.set_xticks(range(0, 41, 5)); eje.legend(); eje.set_title("Distribución del porcentaje de propina")
guardar(fig, "ej4_p8_propinas")
p8b.sort_values("pct_viajes", ascending=False).groupby("tipo_taxi").head(5)

,tipo_taxi,pct_propina,viajes,pct_viajes
61,yellow,20,9387957,51.088
20,green,20,104257,49.804
25,green,25,22954,10.965
0,green,0,19557,9.342
41,yellow,0,1673128,9.105
66,yellow,25,1501055,8.169
30,green,30,10758,5.139
56,yellow,15,734244,3.996
15,green,15,7810,3.731
51,yellow,10,681733,3.710


## P9. Composición del total y consistencia

In [14]:
q("p9_composicion_total")

,tipo_taxi,tarifa,extra,mta_tax,improvement_surcharge,congestion_surcharge,cbd_congestion_fee,airport_fee,ehail_fee,peajes,propina,total,pct_con_cargo_cbd,viajes_total_inconsistente,pct_total_inconsistente
0,yellow,21.29,1.17,0.50,0.98,1.69,0.54,0.13,0.0,0.55,2.88,30.24,72.39,10089345,35.734
1,green,16.76,0.83,0.56,0.92,0.78,0.06,0.00,0.0,0.30,2.60,25.39,8.47,65108,20.158


In [15]:
q("p9b_inconsistencia_por_proveedor")

,tipo_taxi,vendor_id,viajes,inconsistentes,pct_inconsistentes,diferencia_mas_frecuente
0,yellow,1,5374624,4631820,86.2,-3.25
1,yellow,2,22800920,5397888,23.7,2.50
2,yellow,6,59372,59371,100.0,12.20
3,green,1,27000,26145,96.8,-1.00
4,green,2,261148,4127,1.6,2.75
5,green,6,34836,34836,100.0,12.20


## P10. Distribución del monto total y valores atípicos

In [16]:
q("p10a_distribucion_total")

,tipo_taxi,q1,mediana,q3,limite_tukey,p99,maximo,atipicos_tukey,pct_atipicos_tukey,tarifa_por_milla_extrema
0,yellow,17.50,23.58,34.5,60.00,104.96,1065.72,2403726,8.51,4523
1,green,15.12,20.52,29.7,51.57,94.75,668.80,21331,6.60,35


In [17]:
p10b = q("p10b_histograma_total")
fig, eje = plt.subplots(figsize=(9, 3.5))
for tipo, df in p10b.groupby("tipo_taxi"):
    eje.step(df.desde_usd, df.pct_viajes, where="post", label=tipo, color=COLORES[tipo], linewidth=2)
eje.set_xlabel("total_amount (USD, intervalos de $5)"); eje.set_ylabel("% de viajes")
eje.set_title("Distribución del monto total (< $150)"); eje.legend()
guardar(fig, "ej4_p10_total")